# Mision 1: Segmentando el Catalogo para Recomendar

Este notebook agrupa productos de Olist en clusters coherentes usando K-Means,
con PCA para reducir dimensiones y visualizar el resultado. El objetivo es
identificar grupos de productos que puedan servir de base para recomendaciones
(ej: si un cliente compra un producto, recomendar otros del mismo cluster).

Se clusteriza a nivel de **producto**, no de cliente: el 97% de los clientes de
Olist compra una sola vez, por lo que casi no hay variacion real en el
comportamiento de compra para agrupar clientes de forma significativa.

Tablas usadas: order_items, products, category_translation. El detalle de cada
tabla y sus relaciones esta documentado en notebooks/00_inventario_datos.ipynb.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

data_raw = Path("..") / "data" / "raw"

order_items = pd.read_csv(data_raw / "olist_order_items_dataset.csv")
products = pd.read_csv(data_raw / "olist_products_dataset.csv")
category_translation = pd.read_csv(data_raw / "product_category_name_translation.csv")

## Consolidar ventas por producto

Cada fila de order_items es una venta individual del mismo producto. Agregar por
product_id: precio promedio, flete promedio, y cantidad de veces vendido
(proxy de popularidad).

In [2]:
producto_agregado = order_items.groupby('product_id').agg({
    'price': 'mean',
    'freight_value': 'mean',
    'order_id': 'count'
}).reset_index()

producto_agregado.columns = ['product_id', 'precio_promedio', 'flete_promedio', 'veces_vendido']
producto_agregado.head()

,product_id,precio_promedio,flete_promedio,veces_vendido
0,00066f42aeeb9f3007548bb9d3f33c38,101.65,18.59,1
1,00088930e925c41fd95ebfe695fd2655,129.90,13.93,1
2,0009406fd7479715e4bef61dd91f2462,229.00,13.10,1
3,000b8f95fcb9e0096488278317764d19,58.90,19.60,2
4,000d9be29b5207b54e86aa1b1ac54872,199.00,19.27,1


## Anexar ficha tecnica de cada producto

Sumar peso, dimensiones y categoria a cada producto. Calcular volumen a partir
de las tres dimensiones (largo x alto x ancho).

In [3]:
catalogo = producto_agregado.merge(
    products[['product_id', 'product_category_name', 'product_weight_g',
              'product_length_cm', 'product_height_cm', 'product_width_cm']],
    on='product_id'
).merge(
    category_translation,
    on='product_category_name', how='left'
)

catalogo['volumen_cm3'] = (
    catalogo['product_length_cm'] *
    catalogo['product_height_cm'] *
    catalogo['product_width_cm']
)

print(f'Productos en catalogo: {catalogo.shape[0]}')
catalogo.head()

Productos en catalogo: 32951


,product_id,precio_promedio,flete_promedio,veces_vendido,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,volumen_cm3
0,00066f42aeeb9f3007548bb9d3f33c38,101.65,18.59,1,perfumaria,300.0,20.0,16.0,16.0,perfumery,5120.0
1,00088930e925c41fd95ebfe695fd2655,129.90,13.93,1,automotivo,1225.0,55.0,10.0,26.0,auto,14300.0
2,0009406fd7479715e4bef61dd91f2462,229.00,13.10,1,cama_mesa_banho,300.0,45.0,15.0,35.0,bed_bath_table,23625.0
3,000b8f95fcb9e0096488278317764d19,58.90,19.60,2,utilidades_domesticas,550.0,19.0,24.0,12.0,housewares,5472.0
4,000d9be29b5207b54e86aa1b1ac54872,199.00,19.27,1,relogios_presentes,250.0,22.0,11.0,15.0,watches_gifts,3630.0
